# Backtesting Intro – zipline-reloaded

SMA 50/200 crossover, run separately on each ticker in `TICKERS` (each starts with its own $10,000).

In [1]:
%matplotlib inline
import os, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Project root = nearest parent folder that contains src/ (works from any cwd).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

# ---- Tickers to analyse (edit this list) -----------------------------------
TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "NVDA"]

from zipline import run_algorithm
from src.evaluation.csvdir_prep import prepare_csvdir
from src.evaluation.zipline_baselines import (
    ensure_bundle, make_buy_and_hold, make_sma_crossover, trade_pnls,
    utc, show_error,
)
# ---- Data -------------------------------------------------------------------
# A 50/200-day SMA needs years of history, but data/processed/indicators only
# holds ~60 trading days per ticker. So long Yahoo history is downloaded once
# into <root>/data/csvdir/daily/ (cached afterwards).
# Set SOURCE = "indicators" to convert your indicator CSVs instead (short windows only).
DATA_START, DATA_END = "2014-01-01", "2026-10-05"
SOURCE = "yahoo"                       # or "indicators"
CSV_DIR = str(ROOT / "data" / "csvdir")
INDICATORS_DIR = str(ROOT / "data" / "processed" / "indicators")

prepare_csvdir(TICKERS, CSV_DIR, source=SOURCE, start=DATA_START, end=DATA_END,
               indicators_dir=INDICATORS_DIR)
BUNDLE = ensure_bundle(CSV_DIR)        # re-ingests automatically if the tickers change

ticker  rows      first       last
  AAPL  2767 2014-01-02 2024-12-30
  AMZN  2767 2014-01-02 2024-12-30
 GOOGL  2767 2014-01-02 2024-12-30
  META  2767 2014-01-02 2024-12-30
  MSFT  2767 2014-01-02 2024-12-30
  NVDA  2767 2014-01-02 2024-12-30
  TSLA  2767 2014-01-02 2024-12-30


In [2]:
start = utc("2015-01-01")
end   = utc("2026-10-05")   # last bar in the Yahoo download
CAPITAL = 10_000

results = {}
for t in TICKERS:
    initialize, handle_data = make_sma_crossover(t, fast=50, slow=200)
    try:
        results[t] = run_algorithm(
            start=start, end=end,
            initialize=initialize, handle_data=handle_data,
            capital_base=CAPITAL, bundle=BUNDLE,
        )
    except Exception as exc:
        show_error(t, exc)

AAPL: FAILED -> ValueError: Parameter `start` received with timezone defined as 'UTC' although a Date must be timezone naive.
  File "C:\Users\Zamuxolo\AppData\Local\Temp\tmpakpoh9iv\ipykernel_25664\616799783.py", line 9, in <module>
    results[t] = run_algorithm(
                 ^^^^^^^^^^^^^^
  File "c:\Users\Zamuxolo\RAATS_Test\RAATS_Test.venv\Lib\site-packages\zipline\utils\run_algo.py", line 399, in run_algorithm
    return _run(
           ^^^^^
  File "c:\Users\Zamuxolo\RAATS_Test\RAATS_Test.venv\Lib\site-packages\zipline\utils\run_algo.py", line 103, in _run
    if trading_calendar.sessions_distance(start, end) < 1:
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Zamuxolo\RAATS_Test\RAATS_Test.venv\Lib\site-packages\exchange_calendars\exchange_calendar.py", line 2294, in sessions_distance
    start, end = self._parse_start_end_dates(start, end, _parse)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Zamuxolo\RAATS_Test\

In [ ]:
plt.figure(figsize=(12, 6))
rows = []
for t, res in results.items():
    equity = res["portfolio_value"].astype(float)
    plt.plot(equity.index, equity.to_numpy(dtype=float), label=t)
    rows.append({"Ticker": t,
                 "Final value": f"${equity.iloc[-1]:,.2f}",
                 "Total return": f"{equity.iloc[-1] / CAPITAL - 1:.2%}"})
plt.title("Equity Curves - SMA 50/200 Crossover (Zipline)")
plt.ylabel("Portfolio Value ($)")
plt.xlabel("Date")
plt.legend()
plt.grid(True)
plt.show()

display(pd.DataFrame(rows).set_index("Ticker"))